# CS440 Lab 5: Q-learning starter

Adapted from the group's Lab 3 `markov.ipynb`. This is an unfinished starter,
not a trained robot or a completed Lab 5 submission.

Kept: the environment structure, 80/10/10 movement helpers, start lookup,
and a simplified policy display. The start is now top-left for Lab 4/5.

Removed: value iteration, expected-utility calculations, the explicit
transition table used by the planner, old utilities, the policy runner,
and the three living-cost experiments and their saved outputs.

The class name `MDPEnvironment` is retained from Lab 3; it represents the
simulator, not a value-iteration solver. Q-learning will use experiences
without reading the simulator's transition probabilities or reward map.

## Before running

The original notebook used `%run bootstrap.ipynb` and `aima.agents`.
The bootstrap command was removed because that file was not supplied.
The course's AIMA package is now included in `aima/`.
Install dependencies with `python -m pip install -r requirements.txt` in
the notebook's Python environment. The local package supplies
`aima.agents` without the bootstrap notebook.

TODO functions raise
`NotImplementedError` rather than produce misleading results.


In [1]:
# Uses the local course AIMA package; install requirements.txt first.
from aima.agents import *
import random


## Environment

Movement helpers below are adapted from Lab 3. Reward handling is unfinished.
Finish `execute_action` before using this environment for training.

Agreed rules: 80% intended direction, 10% each perpendicular direction;
blocked moves stay in place. Ordinary and blocked moves receive the living
reward. Entering -1 or +10 ends the episode with exactly that reward.


In [2]:
default_world = [['start', 0, 0, 0],
                 [0, 'X', 0, 0],
                 [0, 0,   0, 0],
                 [0, -1,  0, 10]]

class MDPEnvironment(Environment):
    def __init__(self, env=default_world, living_reward=-0.04):
            super().__init__()
            self.status = env
            self.living_reward = living_reward
            self.done = False

    def percept(self, agent):
            """Returns the agent's location, and the location status."""
            return agent.location, self.status[agent.location[0]][agent.location[1]]

    def reset(self, agent):
        """Start a new episode: put the agent on the start cell and clear its score."""
        agent.location = self.default_location(agent)
        agent.performance = 0
        agent.alive = True
        self.done = False
        return agent.location

    def execute_action(self, agent, action):
        """Return (next_location, reward, done) for one action."""
        if self.done:
            raise RuntimeError("Episode is over; call reset(agent) first.")

        if action == 'Right':
            self._attempt(agent, self._right, self._up, self._down)
        elif action == 'Up':
            self._attempt(agent, self._up, self._right, self._left)
        elif action == 'Down':
            self._attempt(agent, self._down, self._left, self._right)
        elif action == 'Left':
            self._attempt(agent, self._left, self._up, self._down)
        else:
            raise ValueError(f"Unknown action: {action}")

        cell = self.status[agent.location[0]][agent.location[1]]
        if cell in (10, -1):
            reward = cell              # exactly +10 or -1, no living reward added
            self.done = True
            agent.alive = (cell == 10)
        else:
            reward = self.living_reward  # ordinary and blocked moves

        agent.performance += reward
        return agent.location, reward, self.done
    
    def _attempt(self, agent, intended_action, accidental_action1, accidental_action2):
        slip = random.random()
        tup = slip < 0.8, slip >= 0.9

        match tup:
            case True, _:
                intended_action(agent)
            case _, False:
                accidental_action1(agent)
            case _, True:
                accidental_action2(agent)

    """ Movements are named so that they correspond to the actual direction.
    For example, an agent at list indices (0, 1)
    could go down to make the new list indices (1, 1),
    or it could go left to make the new list indices (0, 0)"""

    def _down(self, agent):
        newX = agent.location[0] + 1
        Y = agent.location[1]
        if not (newX >= len(self.status) or self.status[newX][Y] == 'X'):
            agent.location = newX, Y

    def _up(self, agent):
        newX = agent.location[0] - 1
        Y = agent.location[1]
        if not (newX < 0 or self.status[newX][Y] == 'X'):
            agent.location = newX, Y

    def _right(self, agent):
        X = agent.location[0]
        newY = agent.location[1] + 1
        if not (newY >= len(self.status[0]) or self.status[X][newY] == 'X'):
            agent.location = X, newY

    def _left(self, agent):
        X = agent.location[0]
        newY = agent.location[1] - 1
        if not (newY < 0 or self.status[X][newY] == 'X'):
            agent.location = X, newY

    def default_location(self, thing):
        for i, list in enumerate(self.status):
            for j, element in enumerate(list):
                if element == 'start':
                    return (i, j)

    def find_recharge_location(self):
        """Return the (row, col) of the cell labeled 10."""
        for i, row in enumerate(self.status):
            for j, element in enumerate(row):
                if element == 10:
                    return (i, j)
        return None

    def __str__(self):
        return str(self.status)


In [ ]:
ACTIONS = ['Up', 'Down', 'Left', 'Right']

LIVING_REWARD = -0.04
GAMMA = 0.95
ALPHA = 0.1
EPSILON = 0.2
EVALUATION_EPSILON = 0.0

# Training and evaluation settings
TRAIN_EPISODES = 2000
EVAL_EPISODES = 500
MAX_STEPS = 100
TRAIN_SEED = 0
EVAL_SEED = 1

## Q-learning

The learner must not read `env.status`, `_attempt`, or recharge-location
helpers to derive a policy. Only the simulator uses those internals.


In [4]:
def choose_action(q_values, state, epsilon, rng):
    """Epsilon-greedy: random action with probability epsilon, else the best one (ties broken randomly)."""
    if rng.random() < epsilon or state not in q_values:
        return rng.choice(ACTIONS)
    best = max(q_values[state].values())
    return rng.choice([a for a in ACTIONS if q_values[state][a] == best])


def outcome_of(reward, done):
    """Label how an episode ended, using only the observed reward."""
    if not done:
        return 'timeout'
    return 'goal' if reward > 0 else 'trap'


def q_learning(env, episodes, max_steps, alpha=ALPHA, gamma=GAMMA,
               epsilon=EPSILON, seed=0):
    """Learn action values from environment interactions. Returns (q_values, results)."""
    random.seed(seed)                  # the environment's slips use the random module
    rng = random.Random(seed)          # the agent's own exploration choices
    robot = Agent(lambda percept: None)
    q_values = {}
    results = []

    for episode in range(episodes):
        state = env.reset(robot)
        q_values.setdefault(state, {a: 0.0 for a in ACTIONS})
        total, steps, reward, done = 0.0, 0, 0.0, False

        while steps < max_steps and not done:
            action = choose_action(q_values, state, epsilon, rng)
            next_state, reward, done = env.execute_action(robot, action)

            if done:
                target = reward        # do not bootstrap from a terminal state
            else:
                q_values.setdefault(next_state, {a: 0.0 for a in ACTIONS})
                target = reward + gamma * max(q_values[next_state].values())

            q_values[state][action] += alpha * (target - q_values[state][action])
            state = next_state
            total += reward
            steps += 1

        results.append({'episode': episode + 1, 'reward': total, 'steps': steps,
                        'outcome': outcome_of(reward, done)})
    return q_values, results


def derive_policy(q_values):
    """Best action for each position the robot visited."""
    return {state: max(ACTIONS, key=lambda a: values[a]) for state, values in q_values.items()}


def evaluate_agent(env, q_values, episodes, max_steps, seed=1, show_paths=3):
    """Evaluate the learned policy without changing Q-values (epsilon = EVALUATION_EPSILON)."""
    random.seed(seed)
    rng = random.Random(seed)
    robot = Agent(lambda percept: None)
    counts = {'goal': 0, 'trap': 0, 'timeout': 0}
    rewards, step_counts = [], []

    for episode in range(episodes):
        state = env.reset(robot)
        path = [state]
        total, steps, reward, done = 0.0, 0, 0.0, False

        while steps < max_steps and not done:
            action = choose_action(q_values, state, EVALUATION_EPSILON, rng)
            next_state, reward, done = env.execute_action(robot, action)
            path.append(f"{action} -> {next_state}")
            state = next_state
            total += reward
            steps += 1

        result = outcome_of(reward, done)
        counts[result] += 1
        rewards.append(total)
        step_counts.append(steps)
        if episode < show_paths:
            print(f"Episode {episode + 1} ({result}, reward {total:.2f}, {steps} steps):")
            print("  " + " | ".join(str(p) for p in path))

    summary = {'episodes': episodes, **counts,
               'success_rate': counts['goal'] / episodes,
               'average_reward': sum(rewards) / episodes,
               'average_steps': sum(step_counts) / episodes}
    print(f"\nEvaluation over {episodes} episodes: {counts}")
    print(f"Success rate: {summary['success_rate']:.1%}  "
          f"Average reward: {summary['average_reward']:.2f}  "
          f"Average steps: {summary['average_steps']:.1f}")
    return summary


## Policy display

Call this helper after deriving a policy from the learned Q-table.
The default world is used here only to display obstacles and terminal cells.


In [5]:
direction = {'Up': 'U', 'Down': 'D', 'Left': 'L', 'Right': 'R'}
def print_policy(policy, world=default_world): 
    print("Policy: ")
    for i in range(len(world)):
        row = []
        for g in range(len(world[0])):
            postion = world[i][g]
            if postion == "X":
                row.append(' X ')
            elif postion == -1:
                row.append('TRAP')
            elif postion == 10:
                row.append('RECHARGE')
            else:
                row.append(direction[policy[(i, g)]] if (i, g) in policy else '?')
        print(' '.join(row))


In [6]:
env = MDPEnvironment(living_reward=LIVING_REWARD)
print("Recharge location:", env.find_recharge_location())

q_values, training = q_learning(env, TRAIN_EPISODES, MAX_STEPS, alpha=ALPHA,
                                gamma=GAMMA, epsilon=EPSILON, seed=TRAIN_SEED)

last = training[-100:]
print("Training, last 100 episodes:",
      {o: sum(r['outcome'] == o for r in last) for o in ('goal', 'trap', 'timeout')},
      f"average reward {sum(r['reward'] for r in last) / len(last):.2f}")

policy = derive_policy(q_values)
print_policy(policy)
print()
evaluation = evaluate_agent(env, q_values, EVAL_EPISODES, MAX_STEPS, seed=EVAL_SEED)

Recharge location: (3, 3)
Training, last 100 episodes: {'goal': 100, 'trap': 0, 'timeout': 0} average reward 9.64
Policy: 
R R R D
U  X  R D
U U R D
L TRAP U RECHARGE

Episode 1 (goal, reward 9.76, 7 steps):
  (0, 0) | Right -> (0, 1) | Right -> (0, 1) | Right -> (0, 2) | Right -> (0, 3) | Down -> (1, 3) | Down -> (2, 3) | Down -> (3, 3)
Episode 2 (goal, reward 9.72, 8 steps):
  (0, 0) | Right -> (0, 1) | Right -> (0, 2) | Right -> (0, 3) | Down -> (0, 2) | Right -> (0, 3) | Down -> (1, 3) | Down -> (2, 3) | Down -> (3, 3)
Episode 3 (goal, reward 9.80, 6 steps):
  (0, 0) | Right -> (0, 1) | Right -> (0, 2) | Right -> (1, 2) | Right -> (2, 2) | Right -> (2, 3) | Down -> (3, 3)

Evaluation over 500 episodes: {'goal': 500, 'trap': 0, 'timeout': 0}
Success rate: 100.0%  Average reward: 9.72  Average steps: 8.1


# Hyperparameter Evaluation

In [ ]:
# Writes results.json so the README and presentation can quote real numbers.
import json
from datetime import datetime

def window_summary(rows):
    return {'episodes': len(rows),
            **{o: sum(r['outcome'] == o for r in rows) for o in ('goal', 'trap', 'timeout')},
            'average_reward': sum(r['reward'] for r in rows) / len(rows),
            'average_steps': sum(r['steps'] for r in rows) / len(rows)}

record = {
    'recorded_at': datetime.now().isoformat(timespec='seconds'),
    'hyperparameters': {'LIVING_REWARD': LIVING_REWARD, 'GAMMA': GAMMA, 'ALPHA': ALPHA,
                        'EPSILON': EPSILON, 'EVALUATION_EPSILON': EVALUATION_EPSILON,
                        'TRAIN_EPISODES': TRAIN_EPISODES, 'EVAL_EPISODES': EVAL_EPISODES,
                        'MAX_STEPS': MAX_STEPS, 'TRAIN_SEED': TRAIN_SEED, 'EVAL_SEED': EVAL_SEED},
    'training_first_100': window_summary(training[:100]),
    'training_last_100': window_summary(training[-100:]),
    'evaluation': evaluation,
    'policy': {f"{r},{c}": a for (r, c), a in sorted(policy.items())},
}
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(record, f, indent=2)

print(json.dumps({k: record[k] for k in ('training_first_100', 'training_last_100', 'evaluation')}, indent=2))
print("Saved results.json")


{
  "training_first_100": {
    "episodes": 100,
    "goal": 88,
    "trap": 12,
    "timeout": 0,
    "average_reward": 8.2228,
    "average_steps": 12.43
  },
  "training_last_100": {
    "episodes": 100,
    "goal": 100,
    "trap": 0,
    "timeout": 0,
    "average_reward": 9.64,
    "average_steps": 10.0
  },
  "evaluation": {
    "episodes": 500,
    "goal": 500,
    "trap": 0,
    "timeout": 0,
    "success_rate": 1.0,
    "average_reward": 9.71752,
    "average_steps": 8.062
  }
}
Saved results.json


## Tests

Run these cells from the project root (the folder containing `scripts/`).
The first checks the Q-learning code in this notebook. The second checks the
agent tooling: the guard hook, the checker hook, and the skill and subagent files.


In [8]:
# Q-learning tests: environment rules, epsilon-greedy, Q-updates, evaluation,
# and an end-to-end training check. Runs from the project root.
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'unittest', '-v', 'scripts.test_q_learning'],
    capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("Q-learning tests:", "PASSED" if result.returncode == 0 else "FAILED")



test_epsilon_one_explores_all_actions (scripts.test_q_learning.ChooseActionTests.test_epsilon_one_explores_all_actions) ... ok
test_exploration_rate_roughly_matches_epsilon (scripts.test_q_learning.ChooseActionTests.test_exploration_rate_roughly_matches_epsilon) ... ok
test_greedy_picks_best_action (scripts.test_q_learning.ChooseActionTests.test_greedy_picks_best_action) ... ok
test_ties_are_broken_randomly_among_best_only (scripts.test_q_learning.ChooseActionTests.test_ties_are_broken_randomly_among_best_only) ... ok
test_unseen_state_falls_back_to_random (scripts.test_q_learning.ChooseActionTests.test_unseen_state_falls_back_to_random) ... ok
test_action_after_episode_end_raises (scripts.test_q_learning.EnvironmentTests.test_action_after_episode_end_raises) ... ok
test_obstacle_is_never_entered (scripts.test_q_learning.EnvironmentTests.test_obstacle_is_never_entered) ... ok
test_ordinary_and_blocked_moves_get_living_reward (scripts.test_q_learning.EnvironmentTests.test_ordinary_and_

In [9]:
# Agent/feature tests: the PreToolUse guard hook and the checker hook, skill, and
# subagent configuration (scripts/check_project.py and .claude/ files).
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'unittest', '-v',
     'scripts.test_guard_tools', 'scripts.test_code_features'],
    capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("Agent/feature tests:", "PASSED" if result.returncode == 0 else "FAILED")



test_bad_input_does_not_block (scripts.test_guard_tools.GuardTests.test_bad_input_does_not_block) ... ok
test_env_file_denied (scripts.test_guard_tools.GuardTests.test_env_file_denied) ... ok
test_env_lookalikes_allowed (scripts.test_guard_tools.GuardTests.test_env_lookalikes_allowed) ... ok
test_lab4_protected (scripts.test_guard_tools.GuardTests.test_lab4_protected) ... ok
test_normal_work_allowed (scripts.test_guard_tools.GuardTests.test_normal_work_allowed) ... ok
test_risky_bash_asks (scripts.test_guard_tools.GuardTests.test_risky_bash_asks) ... ok
test_hook_skips_files_outside_project (scripts.test_code_features.CodeFeatureTests.test_hook_skips_files_outside_project) ... ok
test_missing_skill_fails (scripts.test_code_features.CodeFeatureTests.test_missing_skill_fails) ... ok
test_nonfinite_learning_rate_fails (scripts.test_code_features.CodeFeatureTests.test_nonfinite_learning_rate_fails) ... ok
test_out_of_range_hyperparameter_fails (scripts.test_code_features.CodeFeatureTests.